# Phase 4: Translation QA
Validate that `lyrics_in_en` is predominantly English using FastText language identification.

Pass criteria:
- Pass rate >= 97%
- Confidence threshold >= 0.70

Outputs:
- `data/processed/lyrics_trans_qa_failures.csv`
- `data/processed/lyrics_trans_qa_summary.csv`

In [ ]:
# Install fasttext only if needed
import subprocess
import sys

try:
    import fasttext
except ImportError:
    print("Installing fasttext-wheel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fasttext-wheel", "-q"])
    print("fasttext-wheel installed.")

In [ ]:
from pathlib import Path
import pandas as pd
import fasttext
import fasttext.FastText

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = PROCESSED_DIR / "03_lyrics_trans.csv"
FAILURES_PATH = PROCESSED_DIR / "lyrics_trans_qa_failures.csv"
SUMMARY_PATH = PROCESSED_DIR / "lyrics_trans_qa_summary.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "lid.176.ftz"

TARGET_PASS_RATE = 0.97
CONF_THRESHOLD = 0.70
SNIPPET_CHARS = 200
TRANSLATE_MAX_LENGTH = 5000
REVIEW_MIN_LENGTH = 500

required_cols = ["artist", "title", "spotify_uri", "original_lang", "lyrics_in_en", "length", "translation_review_required"]
df = pd.read_csv(INPUT_PATH)
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in {INPUT_PATH.name}: {missing_cols}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"FastText model not found at {MODEL_PATH}")

ft_model = fasttext.load_model(str(MODEL_PATH))

# Rows flagged for manual review from the same 03_lyrics_trans.csv
review_mask = df["translation_review_required"].fillna(False).astype(bool)
review_rows = int(review_mask.sum())

def make_snippet(text: str, n_chars: int = SNIPPET_CHARS) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    return text[:n_chars].replace("\n", " ").strip()

# QA metric is computed on evaluable rows that are not skipped by max-length gate
snippets = df["lyrics_in_en"].map(make_snippet)
evaluable_mask = snippets.ne("") & (df["length"] < TRANSLATE_MAX_LENGTH)
evaluable_count = int(evaluable_mask.sum())

df_qa = df.copy()
df_qa["qa_pred_lang"] = pd.NA
df_qa["qa_pred_confidence"] = pd.NA
df_qa["qa_is_english"] = pd.NA
df_qa["qa_pass"] = pd.NA

if evaluable_count > 0:
    labels, scores = ft_model.predict(snippets[evaluable_mask].tolist(), k=1)
    pred_lang = [lbl[0].replace("__label__", "") for lbl in labels]
    pred_conf = [float(sc[0]) for sc in scores]

    df_qa.loc[evaluable_mask, "qa_pred_lang"] = pred_lang
    df_qa.loc[evaluable_mask, "qa_pred_confidence"] = pred_conf
    df_qa.loc[evaluable_mask, "qa_is_english"] = [lang == "en" for lang in pred_lang]
    df_qa.loc[evaluable_mask, "qa_pass"] = [
        (lang == "en") and (conf >= CONF_THRESHOLD)
        for lang, conf in zip(pred_lang, pred_conf)
    ]

passed_count = int(df_qa.loc[evaluable_mask, "qa_pass"].fillna(False).sum()) if evaluable_count > 0 else 0
pass_rate = (passed_count / evaluable_count) if evaluable_count > 0 else 0.0

failed_df = df_qa.loc[evaluable_mask & (~df_qa["qa_pass"].fillna(False)), [
    "artist",
    "title",
    "spotify_uri",
    "original_lang",
    "length",
    "translation_review_required",
    "lyrics_in_en",
    "qa_pred_lang",
    "qa_pred_confidence",
]]
failed_df.to_csv(FAILURES_PATH, index=False)

summary_df = pd.DataFrame([
    {"metric": "rows_total", "value": len(df_qa)},
    {"metric": "rows_flagged_for_review", "value": review_rows},
    {"metric": "review_threshold_min_length", "value": REVIEW_MIN_LENGTH},
    {"metric": "translate_max_length", "value": TRANSLATE_MAX_LENGTH},
    {"metric": "rows_evaluable", "value": evaluable_count},
    {"metric": "rows_passed", "value": passed_count},
    {"metric": "pass_rate", "value": pass_rate},
    {"metric": "target_pass_rate", "value": TARGET_PASS_RATE},
    {"metric": "confidence_threshold", "value": CONF_THRESHOLD}
])
summary_df.to_csv(SUMMARY_PATH, index=False)

print(f"Loaded rows: {len(df_qa):,}")
print(f"Rows flagged for manual review (length > {REVIEW_MIN_LENGTH}): {review_rows:,}")
print(f"Evaluable rows (length < {TRANSLATE_MAX_LENGTH} with non-empty lyrics_in_en): {evaluable_count:,}")
print(f"Passed rows: {passed_count:,}")
print(f"Pass rate: {pass_rate:.2%}")
print(f"Target pass rate: {TARGET_PASS_RATE:.2%}")
print(f"Confidence threshold: {CONF_THRESHOLD:.2f}")
print(f"Failures written to: {FAILURES_PATH}")
print(f"Summary written to: {SUMMARY_PATH}")

if pass_rate >= TARGET_PASS_RATE:
    print("QA RESULT: PASS")
else:
    print("QA RESULT: FAIL")

print("\nSample flagged-for-review rows:")
display(df_qa.loc[review_mask, ["spotify_uri", "original_lang", "length", "translation_review_required"]].head(20))

print("\nSample QA failures:")
display(failed_df.head(20))